# GI Mod to Dump Converter
[![Static Badge](https://img.shields.io/badge/Jupyter_Notebook-F37726?style=for-the-badge)](https://jupyter.org/)

<br>

Converts the binary mod files into the collected dump files from a GIMI frame analysis so that the mod can used in Blender

<br>

## Contributors

|   |   |
|---|---|
| **[Albert Gold](https://github.com/Alex-Au1)**  | [![](https://dcbadge.limes.pink/api/shield/367087171154214914?theme=discord-inverted)](https://discordlookup.com/user/367087171154214914) |


<br>

## Requirements
- Python (Version 3.6 or up)

<br>
<br>

## Installation
Choose how to install AGRemap's API

**Option A**: If you want to install through [Pypi](https://pypi.org/project/AnimeGameRemap/), you run the pip install command below

In [ ]:
%pip install -U AnimeGameRemap

In [2]:
import AnimeGameRemap as AGR

<br>

**Option B**: Alternatively, you can locally import the API from a specific git branch

In [ ]:
import sys

# Note: Make sure the path correctly points where the AGRemap's API is located
sys.path.insert(1, r"../../../Anime Game Remap (for all users)/api/src/py")

import FixRaidenBoss2 as AGR

<br>
<br>

## What is Inside a GI Mod?

> ***📝 NOTE:*** <br>
>
> If you do not care about the internal byte level data structure of a mod, you can skip this section and proceed to the [Initialization](#initialization) codeblock
> 

<br>

### Overview

Typical 3dmigoto mods consists of the following parts:
- .ib (index buffer) files
- .vb (vertex buffer) files

These files makeup the [planar graph](https://en.wikipedia.org/wiki/Planar_graph) (a [graph](https://en.wikipedia.org/wiki/Graph_theory) where there exists an arrangement such that its edges will not cross) needed to display the mod within an R^3 vector space.<br>
A special property of the graph within a mod is that **atomically**, every face in the graph is a triangle.

The .ib files define the atomic triangular faces for the graph, while the .vb files define the data for each vertex in the graph.

<br>

For a GI **character**, a .vb file is typically seperated into the following .buf files:
- Position.buf
- Blend.buf
- Texcoord.buf

<br>

> ***📝 NOTE:*** <br>
>
> The above assumption about how a .vb file is seperated may not apply for **non-character** mods
> 

<br>
<br>

What each file does at a high level is summarized below:

| File | Description |
| ---- | ----------- |
| **Position.buf** | Defines the position of a vertex and how light shines on the vertex. |
| **Blend.buf** | Defines which set of vertices a vertex should anchor to. |
| **Texcoord.buf** | Defines how the texture files will map to the vertex. |


<br>

### Internals
Each .buf file is split into *vertex lines*, where each *vertex line* defines the data for a particular vertex in the graph.
All data types within the .buf file use [little endian mode](https://en.wikipedia.org/wiki/Endianness) (MSB is to the right while LSB is to the left)

Below shows the byte level structure for each *vertex line* of the .buf files. <br>
(You can get the same information from reading the *-vb0.txt* dump file)

<br>

#### Position.buf
| Name | Data Type | Description |
| -----| --------- | ----------- |
| Position | 3 Channel Float32 | Where is the vertex located in the vector space (position vector) |
| Normal | 3 Channel Float32 | [Normal vector](https://en.wikipedia.org/wiki/Tangential_and_normal_components) of the position vector |
| Tangent | 4 Channel Float32 | [Tangent vector](https://en.wikipedia.org/wiki/Tangential_and_normal_components) of the position vector |

<br>

#### Blend.buf
| Name | Data Type | Description |
| ---- | --------- | ----------- |
| BlendWeight | 4 channel Float32 | The distribution of how probable a vertex belongs to a particular set of vertices |
| BlendIndices | 4 channel 32-bit Signed Integer | The indices to the set of vertices that the vertex probably belongs to, corresponding to the **BlendWeight** |

<br>

#### Texcoord.buf
| Name | Data Type | Description |
| ---- | --------- | ----------- |
| Colour | 4 channel [8-bit Unsigned Normalized Integer](https://learn.microsoft.com/en-us/windows/win32/direct3d10/d3d10-graphics-programming-guide-resources-data-conversion) | The base colour at the vertex (RGBA) |
| Texcoord | 2 channel Float32 | The location within the first texture file to map onto a vertex |
| ... | ... | ... |
| Texcoordn | 2 channel Float32 | The location within the nth texture file to map onto a vertex |

<br>


#### .ib Files
Each .ib file is split into *face lines*, where each *face line* defines the vertices from the .buf files that make up the particular face.
All data types within the .ib file use [little endian mode](https://en.wikipedia.org/wiki/Endianness) (MSB is to the right while LSB is to the left)

Below shows the byte level structure for each *face line* of the .buf files. <br>
(You can get the same information from reading the *-ib.txt* dump file)

<br>

| Data Type | Description |
| --------- | ----------- |
| 3 channel 32-bit Unsigned Integer | The index of the vertices from the .buf files that make up the particular face |

<br>
<br>

## Initialization
Run the codeblock below to initialize the necessary tools for the conversion process.

In [ ]:
import os
from enum import Enum
from typing import Optional, List


# FileMagic: Some magical values for certain files
class FileMagic(Enum):
    IBProxyHash = "baddbabe"
    VBProxyHash = "b000b135"


# StrClassifiers: Classifiers for string text
class StrClassifiers(Enum):
    IbFileOrder = AGR.AhoCorasickBuilder().build(data = {"Head": 0,
                                                        "Body": 1,
                                                        "Dress": 2,
                                                        "Extra": 3})


class FileService(AGR.FileService):
    @classmethod
    def writeTxt(cls, file: str, data: str):
        with open(file, "w", encoding = AGR.FileEncodings.UTF8.value) as f:
            f.write(data)


# makeVbFile(posPath, blendPath, texcoordPath): Builds the .vb file out of the 3 .buf files that a
#   GI character splits its vertex data across
def makeVbFile(posPath: str, blendPath: str, texcoordPath: str) -> AGR.VbFile:
    positionFile = AGR.PositionFile(posPath)
    blendFile = AGR.BlendFile(blendPath)
    numOfVertices = len(blendFile.data) // blendFile.bytesPerLine

    # How many texture coordinates a mod carries varies, so the Texcoord.buf's own elements are
    #   worked out from how many bytes it devotes to each vertex
    texcoordBytesPerLine = os.path.getsize(texcoordPath) // numOfVertices
    numOfTexcoords = (texcoordBytesPerLine - AGR.BufElementTypes.ColourRGBA.value.size) // 8

    texcoordElements = [AGR.BufElementTypes.ColourRGBA.value]
    texcoordElements += [AGR.BufElementTypes.TextureCoordinateRG.value] * numOfTexcoords
    texcoordFile = AGR.BufFile(texcoordPath, texcoordElements, fileType = "Texcoord")

    # 'merge' stitches the 3 files together line by line and takes on all of their elements
    vbFile = AGR.VbFile(b"", [])
    vbFile.merge([positionFile, blendFile, texcoordFile])
    return vbFile


# ibToDump(ibFile, firstIndex, dstFolder): Converts a .ib file into a dumped ib.txt file
def ibToDump(ibFile: AGR.IbFile, firstIndex: int, dstFolder: Optional[str] = None):
    srcFilePath = AGR.FilePath(ibFile.src)
    if (dstFolder is None):
        dstFolder = srcFilePath.folder

    dstFile = os.path.join(dstFolder, f"{srcFilePath.baseName}-ib={FileMagic.IBProxyHash.value}.txt")
    FileService.writeTxt(dstFile, ibFile.getDumpStr(firstIndex))


# vbToDump(vbFile, dstFolder, modName, modObjs): Converts a .vb file into a dumped vb.txt file
def vbToDump(vbFile: AGR.VbFile, dstFolder: str, modName: str, modObjs: List[str]):
    if (not modObjs):
        return

    result = vbFile.getDumpStr()
    for modObj in modObjs:
        fixedFile = os.path.join(dstFolder, f"{AGR.TextTools.capitalize(modName)}{AGR.TextTools.capitalize(modObj)}-vb0={FileMagic.VBProxyHash.value}.txt")
        FileService.writeTxt(fixedFile, result)

<br>
<br>

## File Setup
Ensure the file paths are set correctly in the data class below in the following constants:
- **ConvertData**

In [12]:
import glob
import os


class ModConvertData():
    def __init__(self, modName: str, modObjs: List[str], srcFolder: str, dstFolder: str):
        self.modName = modName
        self.modObjs = modObjs
        self.srcFolder = srcFolder
        self.dstFolder = dstFolder

        self.ibFiles = []
        self.positionFile = None
        self.blendFile = None
        self.texcoordFile = None

    def getIbFileOrder(self, ibFile: str):
        ibBasename = os.path.basename(ibFile)
        modObj, modObjOrder = StrClassifiers.IbFileOrder.value.getMaximal(ibBasename, errorOnNotFound = False)
        
        if (modObjOrder is None):
            return 999
        return modObjOrder

    def readFiles(self):
        self.ibFiles = glob.glob(os.path.join(self.srcFolder, "*.ib"))
        self.ibFiles.sort(key = self.getIbFileOrder)

        blendFiles = glob.glob(os.path.join(self.srcFolder, "*RemapBlend*.buf"))
        positionFiles = glob.glob(os.path.join(self.srcFolder, "*Position*.buf"))
        texcoordFiles = glob.glob(os.path.join(self.srcFolder, "*Texcoord*.buf"))

        if (blendFiles):
            self.blendFile = blendFiles[0]

        if (positionFiles):
            self.positionFile = positionFiles[0]

        if (texcoordFiles):
            self.texcoordFile = texcoordFiles[0]

    def convert(self):
        currentIndex = 0
        for ibFilePath in self.ibFiles:
            ibFile = AGR.IbFile(ibFilePath)
            ibToDump(ibFile, currentIndex, dstFolder = self.dstFolder)
            currentIndex += ibFile.getIndexCount()

        if (self.positionFile is None or self.blendFile is None or self.texcoordFile is None):
            return

        vbFile = makeVbFile(self.positionFile, self.blendFile, self.texcoordFile)
        vbToDump(vbFile, self.dstFolder, self.modName, self.modObjs)


#######################
# Check if the file paths here are set correctly

ConvertData = [
    #ModConvertData("HuTao", ["Head", "Body"], r"E:\Computer\Games\Wuthering Waves Mods\Importer\GIMI\Mods\Hutao6", r"E:\Computer\Games\Wuthering Waves Mods\Importer\GIMI\Mods\Hutao6"),
    #ModConvertData("Ayaka", ["Head", "Body", "Dress"], r"E:\Computer\Games\Wuthering Waves Mods\Importer\GIMI\Mods\Ayaka6\ayaka_swimsuit_v2\AyakaModf0001", r"E:\Computer\Games\Wuthering Waves Mods\Importer\GIMI\Mods\Ayaka6\ayaka_swimsuit_v2\AyakaModf0001"),
    #ModConvertData("Keqing", ["Head", "Body", "Dress"], r"E:\Computer\Games\Wuthering Waves Mods\Importer\GIMI\Keqing\keqing_first_latern_rite\0 - keqing_firstlanternrite", r"E:\Computer\Games\Wuthering Waves Mods\Importer\GIMI\Keqing\keqing_first_latern_rite\0 - keqing_firstlanternrite"),
    #ModConvertData("RaidenShogun", ["Head", "Body", "Dress"], r"E:\Computer\Games\Genshin\Mods\Characters\Ei\raiden_winter_princess\0 - OGColors\RaidenShogunModOG", r"E:\Computer\Games\Genshin\Mods\Characters\Ei\raiden_winter_princess\0 - OGColors\RaidenShogunModOG")
    #ModConvertData("TravelerGirl", ["Head", "Body", "Dress"], r"E:\Computer\Games\Wuthering Waves Mods\Importer\Prod_Mods\Lumine\lumine_latern_rite", r"E:\Computer\Games\Wuthering Waves Mods\Importer\Prod_Mods\Lumine\lumine_latern_rite")
    #ModConvertData("Ayaka", ["Head", "Body", "Dress"], r"E:\Computer\Games\Wuthering Waves Mods\Importer\GIMI\Ayaka2\ayaka_hyoukai\ayaka_hyoukai\Ayaka in Hyoukai Sonata", r"E:\Computer\Games\Wuthering Waves Mods\Importer\GIMI\Ayaka2\ayaka_hyoukai\ayaka_hyoukai\Ayaka in Hyoukai Sonata")
    #ModConvertData("KaeyaSailwind", ["Head", "Body", "Dress"], r"E:\Computer\Games\Wuthering Waves Mods\Importer\GIMI\Mods\KaeyaSailwind2\kaeyasailwindpants\KaeyaSailwindMod", r"E:\Computer\Games\Wuthering Waves Mods\Importer\GIMI\Mods\KaeyaSailwind2\kaeyasailwindpants\KaeyaSailwindMod")
    #ModConvertData("AyakaSpringBloom", ["Head", "Body", "Dress"], r"E:\Computer\Games\Wuthering Waves Mods\Importer\GIMI\AyakaSpringBloom7\ayaka dress", r"E:\Computer\Games\Wuthering Waves Mods\Importer\GIMI\AyakaSpringBloom7\ayaka dress")
    ModConvertData("ShenheFrostFlower", ["Head", "Body", "Dress", "Extra"], r"E:\Computer\Games\Wuthering Waves Mods\Importer\GIMI\Mods\ShenheFrostFlower4\ShenheFrostFlowerHalf\ShenheFrostFlowerHalf", r"E:\Computer\Downloads\ShenheFrostFlowerRemap2")
]

#######################

<br>
<br>

## Run the Converter
The code block below converts the binary mod files to their corresponding dump files

In [13]:
heading = AGR.Heading(sideLen = 10, sideChar = "=")

for convertDatum in ConvertData:
    print(f"{heading.close()}\n")

    print(f"Mod: {convertDatum.srcFolder}")
    print(f"Reading files...")
    convertDatum.readFiles()

    print(f"Converting files...")
    convertDatum.convert()

    print(f"\n{heading.close()}")


Mod: E:\Computer\Games\Wuthering Waves Mods\Importer\GIMI\Mods\ShenheFrostFlower4\ShenheFrostFlowerHalf\ShenheFrostFlowerHalf
Reading files...
Converting files...

